In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


In [ ]:

df = pd.read_csv('Dataset_spam.csv')
print(df.head())


      LABEL                                               TEXT URL EMAIL PHONE
0       ham  Hi! You just spoke to DEEPAK. We'd like to kno...  No    No    No
1       ham                  Yay can't wait to party together!  No    No    No
2       ham                       At what time are you coming.  No    No    No
3  smishing  Dear customer your PAY2TMKYC has been expired,...  No    No   Yes
4       ham  Yo you around? A friend of mine's lookin to pi...  No    No    No


In [17]:
df.info()

df.shape
df.columns

<class 'pandas.DataFrame'>
RangeIndex: 10191 entries, 0 to 10190
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   LABEL         10191 non-null  str  
 1   TEXT          10191 non-null  str  
 2   URL           10191 non-null  str  
 3   EMAIL         10191 non-null  str  
 4   PHONE         10191 non-null  str  
 5   label_binary  10191 non-null  int64
dtypes: int64(1), str(5)
memory usage: 477.8 KB


Index(['LABEL', 'TEXT', 'URL', 'EMAIL', 'PHONE', 'label_binary'], dtype='str')

In [12]:
df.describe()
df['LABEL'].value_counts()

LABEL
ham         3397
smishing    3397
spam        3397
Name: count, dtype: int64

In [24]:
df.dropna(inplace=True)
df.isnull().sum()


LABEL           0
TEXT            0
URL             0
EMAIL           0
PHONE           0
label_binary    0
dtype: int64

In [25]:
df.drop_duplicates(inplace=True)
df.shape

(8022, 6)

In [ ]:
df['label_binary'] = df['LABEL'].apply(lambda x: 0 if x == 'ham' else 1)
df['label_binary'].value_counts()
#print(sum(df['label_binary'] == 0))

label_binary
1    6794
0    3397
Name: count, dtype: int64

In [29]:
df.head()
if 'label' in df.columns and 'message' in df.columns:
    df = df[['label', 'message']]

print(df.head())

      LABEL                                               TEXT URL EMAIL  \
0       ham  Hi! You just spoke to DEEPAK. We'd like to kno...  No    No   
1       ham                  Yay can't wait to party together!  No    No   
2       ham                       At what time are you coming.  No    No   
3  smishing  Dear customer your PAY2TMKYC has been expired,...  No    No   
4       ham  Yo you around? A friend of mine's lookin to pi...  No    No   

  PHONE  label_binary  
0    No             0  
1    No             0  
2    No             0  
3   Yes             1  
4    No             0  


In [39]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re
import string

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rahad\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rahad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    if pd.isna(text):
        return ""
    
    # Convert to string and lowercase
    text = str(text).lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove special characters and digits (keep only letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def clean_text_with_stemming(text):
    text = clean_text(text)
    words = text.split()
    words = [stemmer.stem(word) for word in words]
    return ' '.join(words)

df['cleaned_text'] = df['TEXT'].apply(clean_text)
df['cleaned_text_stemmed'] = df['TEXT'].apply(clean_text_with_stemming)

print("\nOriginal message sample:")
print(df['TEXT'].iloc[0])
print("\nCleaned message:")
print(df['cleaned_text'].iloc[0])


Original message sample:
Hi! You just spoke to DEEPAK. We'd like to know if you were satisfied with the experience. Reply Toll Free with Yes or No.

Cleaned message:
hi you just spoke to deepak wed like to know if you were satisfied with the experience reply toll free with yes or no


In [45]:
df['char_length'] = df['TEXT'].apply(len)
df['word_count'] = df['cleaned_text'].apply(lambda x: len(x.split()))
df['word_count'].head()
df['char_length']

0        122
1         33
2         28
3        150
4         65
        ... 
10168    163
10171    146
10172    152
10175    146
10189    117
Name: char_length, Length: 8022, dtype: int64

In [ ]:
print("\nMessage length statistics by label:")
print(df.groupby('LABEL')['char_length'].describe())